**Project 1**

**Setting up data:**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import (MinMaxScaler,MaxAbsScaler, RobustScaler)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


features = [
    "matchId",
    "blueTeamControlWardsPlaced", "blueTeamWardsPlaced", "blueTeamTotalKills", "blueTeamDragonKills",
    "blueTeamHeraldKills", "blueTeamTowersDestroyed", "blueTeamInhibitorsDestroyed",
    "blueTeamTurretPlatesDestroyed", "blueTeamFirstBlood", "blueTeamMinionsKilled",
    "blueTeamJungleMinions", "blueTeamTotalGold", "blueTeamXp", "blueTeamTotalDamageToChamps",
    "redTeamControlWardsPlaced", "redTeamWardsPlaced", "redTeamTotalKills", "redTeamDragonKills",
    "redTeamHeraldKills", "redTeamTowersDestroyed", "redTeamInhibitorsDestroyed",
    "redTeamTurretPlatesDestroyed", "redTeamMinionsKilled", "redTeamJungleMinions",
    "redTeamTotalGold", "redTeamXp", "redTeamTotalDamageToChamps",
    "blueWin", "temp"
]

data = pd.read_csv('data.csv', names = features, na_values=["?", "N/A", ""], skipinitialspace=True, skiprows=1)

data_without_classification = data.drop(columns=["matchId","blueWin", "temp"])
results_column = data["blueWin"]



X_train, X_test, y_train, y_test = train_test_split(
    data_without_classification, results_column,
    test_size=0.20,
    stratify=results_column,
    random_state=42,
)

print("train:", len(X_train))
print("test:", len(X_test))

train: 19380
test: 4845


**Benchmark we will compare our model with**

In [ ]:
model1 = DecisionTreeClassifier()
model1.fit(X_train, y_train)

accuracy = model1.score(X_test, y_test)
print(f"Model Test Accuracy: {accuracy * 100:.2f}%")
print(f"Train: {model1.score(X_train, y_train) * 100:.2f}%")
print(f"Test:  {model1.score(X_test, y_test) * 100:.2f}%")

**Data Representation**

We don't have imbalanced data because of this:

In [ ]:
data['blueWin'].value_counts(normalize=True)

**Missing Values**

No missing values:

In [ ]:
data.isna().sum()

**Model Engineering / K Nearest neighbours**

In [ ]:
pipe = Pipeline([("scaling_methods", StandardScaler()), ("knn", KNeighborsClassifier())])

params = {
    "scaling_methods": [
        StandardScaler(),
        MinMaxScaler(),
        MaxAbsScaler(),
         RobustScaler(),
        "passthrough"
    ],
    "knn__n_neighbors": [3, 6, 8, 12, 18, 30, 40, 50,60,70,80, 90 , 150, 200],
    "knn__weights": ["uniform", "distance"]
}

grid1 = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy"
)

grid1.fit(X_train, y_train)

print("Best parameters:")
print(grid1.best_params_)

print("Best cross-validation accuracy:")
print(grid1.best_score_)

predictions = grid1.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

**Model Engineering / Logistic Regression // Without Scaling**

In [4]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear']
}

grid2 = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5)
grid2.fit(X_train, y_train)

predictions = grid2.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.7643


**Model Engineering / Logistic Regression // With Scaling**

In [5]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

param_grid = {
    "logistic__C": [0.01, 0.1, 1, 10, 100],
    "logistic__solver": ["liblinear"]
}

grid2 = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid2.fit(X_train, y_train)

predictions = grid2.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Best parameters:", grid2.best_params_)
print("Best CV accuracy:", grid2.best_score_)
print(f"Test accuracy: {accuracy:.4f}")

model = grid2.best_estimator_.named_steps["logistic"]

importance = pd.Series(
    np.abs(model.coef_[0]),
    index=X_train.columns
).sort_values()

print(importance)

Best parameters: {'logistic__C': 0.01, 'logistic__solver': 'liblinear'}
Best CV accuracy: 0.7586171310629515
Test accuracy: 0.7649
temp                            0.000000
blueTeamMinionsKilled           0.002332
blueTeamFirstBlood              0.011130
blueTeamTowersDestroyed         0.011334
blueTeamWardsPlaced             0.014458
blueTeamControlWardsPlaced      0.015684
redTeamHeraldKills              0.023645
redTeamInhibitorsDestroyed      0.024143
redTeamWardsPlaced              0.028988
redTeamControlWardsPlaced       0.043705
blueTeamInhibitorsDestroyed     0.054428
blueTeamTotalDamageToChamps     0.064086
redTeamTowersDestroyed          0.064980
redTeamTurretPlatesDestroyed    0.082571
redTeamTotalDamageToChamps      0.093055
redTeamDragonKills              0.153998
redTeamTotalKills               0.156475
blueTeamTotalKills              0.188903
blueTeamDragonKills             0.239867
redTeamXp                       0.284460
blueTeamXp                      0.312940
blueTeam

**Logistic Regression // With Scaling // Deleting Weak Features**

In [7]:
drop_features = [
    "redTeamJungleMinions",
    "blueTeamJungleMinions",
    "blueTeamHeraldKills",
    "blueTeamTurretPlatesDestroyed",
    "redTeamMinionsKilled"
]

X = data.drop(columns=["blueWin", "matchId", "temp"])
y = data["blueWin"]

X_reduced = X.drop(columns=drop_features)

X_train, X_test, y_train, y_test = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

param_grid = {
    "logistic__C": [0.01, 0.1, 1, 10, 100],
    "logistic__solver": ["liblinear"]
}

grid2 = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid2.fit(X_train, y_train)

predictions = grid2.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Best parameters:", grid2.best_params_)
print("Best CV accuracy:", grid2.best_score_)
print(f"Test accuracy: {accuracy:.4f}")

Remaining features:
['blueTeamControlWardsPlaced', 'blueTeamWardsPlaced', 'blueTeamTotalKills', 'blueTeamDragonKills', 'blueTeamTowersDestroyed', 'blueTeamInhibitorsDestroyed', 'blueTeamFirstBlood', 'blueTeamMinionsKilled', 'blueTeamTotalGold', 'blueTeamXp', 'blueTeamTotalDamageToChamps', 'redTeamControlWardsPlaced', 'redTeamWardsPlaced', 'redTeamTotalKills', 'redTeamDragonKills', 'redTeamHeraldKills', 'redTeamTowersDestroyed', 'redTeamInhibitorsDestroyed', 'redTeamTurretPlatesDestroyed', 'redTeamTotalGold', 'redTeamXp', 'redTeamTotalDamageToChamps']
Best parameters: {'logistic__C': 0.01, 'logistic__solver': 'liblinear'}
Best CV accuracy: 0.7586171310629515
Test accuracy: 0.7649


**Model Engineering / Decission Tree**

In [ ]:
pipe = Pipeline([("dt", DecisionTreeClassifier())])

params = {
    "dt__criterion": ["gini", "entropy"],
    "dt__max_depth": [None, 3, 5, 10, 15, 20],
    "dt__min_samples_split": [2, 5, 10, 20],
    "dt__min_samples_leaf": [1, 2, 5, 10],
    "dt__max_features": [None, "sqrt", "log2"]
}

grid3 = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid3.fit(X_train, y_train)

print("Best parameters:")
print(grid3.best_params_)

print("Best cross-validation accuracy:")
print(grid3.best_score_)

predictions = grid3.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

**Model Engineering / AdaBoost**

In [ ]:
pipe = Pipeline([("ada", AdaBoostClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42))])

params = {
    "ada__n_estimators": [25, 50, 100, 200, 500],
    "ada__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "ada__estimator__max_depth": [1, 2, 3],
    "ada__estimator__class_weight": [None, "balanced"]
}

grid4 = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

grid4.fit(X_train, y_train)

print("Best parameters:")
print(grid4.best_params_)

print("Best cross-validation accuracy:")
print(grid4.best_score_)

predictions = grid4.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

**Model Engineering / XGBoost**

In [ ]:
pipe = Pipeline([
    ("xgb", XGBClassifier(
        n_estimators=2,
        max_depth=2,
        learning_rate=1,
        objective="binary:logistic"
    ))
])

params = {
    "xgb__n_estimators": [100, 200, 300, 500],
    "xgb__learning_rate": [0.01, 0.05, 0.1, 0.3],
    "xgb__max_depth": [2, 3, 4, 6, 8],
    "xgb__subsample": [0.6, 0.8, 1.0]
}

grid5 = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid5.fit(X_train, y_train)

print("Best parameters:")
print(grid5.best_params_)

print("Best cross-validation accuracy:")
print(grid5.best_score_)

predictions = grid5.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

**Model Engineering / RandomForest**

In [ ]:
pipe = Pipeline([
    ("rf", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ))
])

params = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_features": ["sqrt", "log2", 0.3, 0.5],
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_leaf": [1, 3, 5, 10]
}

grid6 = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid6.fit(X_train, y_train)

print("Best parameters:")
print(grid6.best_params_)

print("Best cross-validation accuracy:")
print(grid6.best_score_)

predictions = grid6.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy: {accuracy:.4f}")

In [ ]:
rf = grid6.best_estimator_.named_steps["rf"]

importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)